# Group 32 — Rohith's ROME single-edit baseline

**Status: prepared, GPU execution pending.** Run cells from top to bottom.

Deliverable: 200 independent GPT-2 XL / CounterFact edits, efficacy, paraphrase and neighborhood scores, raw results and reproducible configuration.

Plan: hardware check → install → persistent storage → 1 fact → 5 facts → 200 facts → export.

The paper's **100.0 / 96.4 / 75.4** values are GPT-2 XL results on **7,500** examples (Table 4). This 200-example subset is a smaller replication, and exact agreement is not expected. [Paper](https://arxiv.org/pdf/2202.05262) · [Official code](https://github.com/kmeng01/rome)

Use **Runtime → Change runtime type → GPU**. A GPU allocation and sufficient compute units are required. No paid upgrade is automated. Existing notebooks are unaffected.


In [3]:
import sys
import subprocess
import shutil
from pathlib import Path

import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not connected. Select Runtime → Change runtime type → GPU."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory GiB:",
    round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1),
)

subprocess.run(["nvidia-smi"], check=True)

free_disk_gib = shutil.disk_usage("/content").free / 2**30
print("Free disk GiB:", round(free_disk_gib, 1))

if free_disk_gib < 12:
    raise RuntimeError("At least 12 GiB of free disk space is required.")

print("Hardware check complete. Share this output before running the installation cell.")


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory GiB: 14.6
Free disk GiB: 193.3
Hardware check complete. Share this output before running the installation cell.


## Install an isolated environment and pin the official source
Colab's GPU PyTorch is reused. Other packages are installed into a separate environment. If installation fails, stop at the error rather than continuing with a partially installed environment. Source is pinned; no ROME algorithm files are patched.


In [7]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/group32_rome")
PYTHON = "/content/group32_rome_py311/bin/python"

# Use a display backend available in the separate environment.
os.environ["MPLBACKEND"] = "Agg"

def run(command, cwd=None):
    result = subprocess.run(
        command,
        cwd=cwd,
        env=os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    print(result.stdout)
    result.check_returncode()

run([
    PYTHON, "-c",
    """
import torch
import numpy
import pyarrow
import pandas
import matplotlib

from rome import apply_rome_to_model
from experiments.py.eval_utils_counterfact import (
    compute_rewrite_quality_counterfact
)

assert torch.cuda.is_available(), "GPU not connected"

x = torch.ones((32, 32), device="cuda")
assert (x @ x)[0, 0].item() == 32.0

print("NumPy:", numpy.__version__)
print("PyArrow:", pyarrow.__version__)
print("Pandas:", pandas.__version__)
print("Matplotlib backend:", matplotlib.get_backend())
print("GPU:", torch.cuda.get_device_name(0))
print("SETUP COMPLETE")
"""
], cwd=REPO)

NumPy: 1.26.4
PyArrow: 17.0.0
Pandas: 2.2.3
Matplotlib backend: Agg
GPU: Tesla T4
SETUP COMPLETE



## Save results persistently
With `USE_DRIVE = True`, Colab will ask you to authorize Drive access. Results go to `MyDrive/Group32/ROME_baseline/gpt2xl_cf200_v1`. Use a new run name if the recorded configuration changes. If Drive is disabled, download the ZIP before the runtime expires.


In [8]:
USE_DRIVE = True
RUN_NAME = 'gpt2xl_cf200_v1'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/Group32/ROME_baseline')
else:
    BASE = Path('/content/Group32/ROME_baseline')
OUT = BASE / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
print('Results:', OUT)


Mounted at /content/drive
Results: /content/drive/MyDrive/Group32/ROME_baseline/gpt2xl_cf200_v1


## Install the experiment wrapper
Uses official ROME and probability evaluation. Restores original weights after every edit, including exceptions. Evaluation processes four prefixes at a time to limit memory. Covariance statistics must download successfully; no silent large recomputation.

The embedded files make this notebook self-contained. The runner saves each completed case atomically and rejects incompatible checkpoints. It does not skip unsuccessful edits or report invented results.


In [10]:
import json
EMBEDDED_FILES = {'baseline_support.py': '"""Result validation and aggregation; no GPU dependencies."""\nimport json\nimport math\nimport os\nfrom pathlib import Path\n\nMETRICS = {\'efficacy\': \'rewrite_prompts_probs\', \'paraphrase\': \'paraphrase_prompts_probs\', \'neighborhood\': \'neighborhood_prompts_probs\'}\nPAPER = {\'efficacy\': 100.0, \'paraphrase\': 96.4, \'neighborhood\': 75.4}\n\ndef atomic_json(path, value):\n    path = Path(path)\n    temp = path.with_suffix(path.suffix + \'.tmp\')\n    with temp.open(\'w\') as f:\n        json.dump(value, f, indent=2, allow_nan=False)\n        f.flush()\n        os.fsync(f.fileno())\n    temp.replace(path)\n\ndef scores(result, phase):\n    out = {}\n    for metric, key in METRICS.items():\n        entries = result[phase][key]\n        if not entries:\n            raise ValueError(f\'Empty {phase}/{key}\')\n        successes = []\n        for pair in entries:\n            true, new = pair[\'target_true\'], pair[\'target_new\']\n            if not all(math.isfinite(x) for x in (true, new)):\n                raise ValueError(\'Nonfinite evaluation probability\')\n            successes.append(true < new if metric == \'neighborhood\' else new < true)\n        out[metric] = 100 * sum(successes) / len(successes)\n    return out\n\ndef validate_result(result, record, run_key):\n    if result[\'run_key\'] != run_key or result[\'case_id\'] != record[\'case_id\']:\n        raise ValueError(\'Result belongs to another run/case\')\n    if result[\'requested_rewrite\'] != record[\'requested_rewrite\']:\n        raise ValueError(\'Requested rewrite mismatch\')\n    if result.get(\'weights_restored\') is not True:\n        raise ValueError(\'Weight restoration was not verified\')\n    for phase in (\'pre\', \'post\'):\n        scores(result, phase)\n    return result\n\ndef aggregate(results):\n    if not results:\n        raise ValueError(\'No results; cannot report scores\')\n    ids = [r[\'case_id\'] for r in results]\n    if len(set(ids)) != len(ids):\n        raise ValueError(\'Duplicate cases\')\n    output = {\'num_cases\': len(results), \'case_ids\': ids}\n    for phase in (\'pre\', \'post\'):\n        rows = [scores(r, phase) for r in results]\n        output[phase] = {m: sum(row[m] for row in rows) / len(rows) for m in METRICS}\n    output[\'paper_gpt2_xl_7500_cases\'] = PAPER\n    output[\'difference_percentage_points\'] = {m: output[\'post\'][m] - PAPER[m] for m in METRICS}\n    return output\n', 'run_baseline.py': '"""Run from the pinned official ROME checkout. Uses original editing/evaluation code."""\nimport argparse\nimport contextlib\nimport hashlib\nimport importlib.metadata\nimport json\nimport platform\nimport random\nimport subprocess\nimport time\nfrom pathlib import Path\n\nfrom baseline_support import atomic_json, aggregate, validate_result\n\nROME_COMMIT = \'0874014cd9837e4365f3e6f3c71400ef11509e04\'\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--limit\', type=int, choices=[1, 5, 200], required=True)\n    parser.add_argument(\'--output\', type=Path, required=True)\n    args = parser.parse_args()\n    import numpy as np\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    from huggingface_hub import model_info\n    from dsets import CounterFactDataset\n    from rome import ROMEHyperParams, apply_rome_to_model\n    import rome.rome_main as rome_main\n    from util import nethook\n    from util.globals import DATA_DIR, STATS_DIR, REMOTE_ROOT_URL\n    import experiments.py.eval_utils_counterfact as evaluation\n\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'A CUDA GPU runtime is required. Choose GPU in Colab runtime settings.\')\n    commit = subprocess.check_output([\'git\', \'rev-parse\', \'HEAD\'], text=True).strip()\n    if commit != ROME_COMMIT:\n        raise RuntimeError(\'ROME version mismatch\')\n    if subprocess.check_output([\'git\', \'diff\', \'--name-only\'], text=True).strip():\n        raise RuntimeError(\'Tracked ROME source files were modified\')\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    seed = 32\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    torch.cuda.manual_seed_all(seed)\n    output = args.output.resolve()\n    output.mkdir(parents=True, exist_ok=True)\n    manifest_path = output / \'manifest.json\'\n    previous = json.loads(manifest_path.read_text()) if manifest_path.exists() else None\n    model_id = \'gpt2-xl\'\n    revision = previous[\'model_revision\'] if previous else model_info(model_id).sha\n    hp_path = Path(\'hparams/ROME/gpt2-xl.json\')\n    hp = json.loads(hp_path.read_text())\n    hparams = ROMEHyperParams.from_json(hp_path)\n    ds = CounterFactDataset(DATA_DIR, size=200)\n    if len(ds) != 200:\n        raise RuntimeError(\'Expected exactly 200 dataset records\')\n    records = list(ds)\n    ids = [r[\'case_id\'] for r in records]\n    if len(set(ids)) != 200:\n        raise RuntimeError(\'Duplicate dataset case IDs\')\n    packages = {x: importlib.metadata.version(x) for x in [\'torch\',\'transformers\',\'tokenizers\',\'datasets\',\'numpy\',\'scipy\',\'scikit-learn\',\'huggingface-hub\']}\n    manifest = {\n        \'runner_sha256\': hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n        \'support_sha256\': hashlib.sha256(Path(__file__).with_name(\'baseline_support.py\').read_bytes()).hexdigest(),\n        \'schema\': 1, \'rome_commit\': commit, \'model\': model_id, \'model_revision\': revision,\n        \'dtype\': \'float32\', \'attention\': \'eager\', \'seed\': seed, \'hparams\': hp,\n        \'selection\': \'first 200 CounterFact records, in published order; no success filtering\',\n        \'case_ids\': ids, \'dataset_sha256\': hashlib.sha256((Path(DATA_DIR)/\'counterfact.json\').read_bytes()).hexdigest(),\n        \'packages\': packages, \'python\': platform.python_version(),\n        \'gpu\': torch.cuda.get_device_name(0), \'cuda\': torch.version.cuda,\n        \'metrics\': \'official probability-based ES/PS/NS; generation metrics omitted\',\n        \'evaluation_prefix_batch_size\': 4,\n    }\n    if previous and previous != manifest:\n        raise RuntimeError(\'Environment/config changed. Use a NEW output folder to avoid mixing results.\')\n    atomic_json(manifest_path, manifest)\n    run_key = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()\n    pending = []\n    for record in records[:args.limit]:\n        path = output / f"case_{record[\'case_id\']}.json"\n        if path.exists():\n            validate_result(json.loads(path.read_text()), record, run_key)\n        else:\n            pending.append(record)\n    if pending:\n        # Download official covariance first. Never silently launch 100k-document recomputation.\n        rel = f"gpt2-xl/wikipedia_stats/transformer.h.17.mlp.c_proj_float32_mom2_100000.npz"\n        stats_path = Path(STATS_DIR) / rel\n        stats_path.parent.mkdir(parents=True, exist_ok=True)\n        if not stats_path.exists():\n            temp = stats_path.with_suffix(\'.download\')\n            torch.hub.download_url_to_file(f\'{REMOTE_ROOT_URL}/data/stats/{rel}\', temp)\n            with np.load(temp, allow_pickle=False) as z:\n                assert z.files, \'Empty covariance archive\'\n            temp.replace(stats_path)\n        model = AutoModelForCausalLM.from_pretrained(model_id, revision=revision,\n            torch_dtype=torch.float32, attn_implementation=\'eager\', use_safetensors=True).eval().cuda()\n        tok = AutoTokenizer.from_pretrained(model_id, revision=revision)\n        tok.pad_token = tok.eos_token\n        # Freeze model parameters; ROME optimizes an internal activation, not these weights.\n        nethook.set_requires_grad(False, model)\n        # Keep original probability calculations; bound the number of prompts per forward pass.\n        original_prediction = evaluation.test_batch_prediction\n        def chunked_prediction(model, tok, prefixes, target_new, target_true):\n            results = []\n            for start in range(0, len(prefixes), 4):\n                results.extend(original_prediction(model, tok, prefixes[start:start+4], target_new, target_true))\n            return results\n        evaluation.test_batch_prediction = chunked_prediction\n        context_path = output / \'context_templates.json\'\n        if context_path.exists():\n            rome_main.CONTEXT_TEMPLATES_CACHE = json.loads(context_path.read_text())\n        else:\n            if list(output.glob(\'case_*.json\')):\n                raise RuntimeError(\'Context templates are missing for an existing run\')\n            templates = rome_main.get_context_templates(model, tok, hparams.context_template_length_params)\n            atomic_json(context_path, templates)\n        atomic_json(output / \'environment.json\', manifest)\n        subprocess.run([\'python\', \'-m\', \'pip\', \'freeze\'], stdout=(output / \'packages.txt\').open(\'w\'), check=True)\n        weights = {f\'{hparams.rewrite_module_tmp.format(layer)}.weight\':\n                   nethook.get_parameter(model, f\'{hparams.rewrite_module_tmp.format(layer)}.weight\')\n                   for layer in hparams.layers}\n        originals = {name: value.detach().cpu().clone() for name, value in weights.items()}\n        for record in pending:\n            case_id = record[\'case_id\']\n            # Per-case seed plus persisted context templates makes resuming reproducible.\n            random.seed(seed + case_id)\n            np.random.seed(seed + case_id)\n            torch.manual_seed(seed + case_id)\n            torch.cuda.manual_seed_all(seed + case_id)\n            started = time.perf_counter()\n            with (output / f\'case_{case_id}.log\').open(\'w\') as log, contextlib.redirect_stdout(log):\n                pre = evaluation.compute_rewrite_quality_counterfact(model, tok, record, None, None)\n                try:\n                    edit_start = time.perf_counter()\n                    edited, _ = apply_rome_to_model(model, tok, [record[\'requested_rewrite\']], hparams,\n                                                   copy=False, return_orig_weights=False)\n                    edit_seconds = time.perf_counter() - edit_start\n                    post = evaluation.compute_rewrite_quality_counterfact(edited, tok, record, None, None)\n                finally:\n                    # Restore even if editing or evaluation raises an exception.\n                    with torch.no_grad():\n                        for name, value in weights.items():\n                            value.copy_(originals[name].to(value.device))\n                    for name, value in weights.items():\n                        if not torch.equal(value.detach().cpu(), originals[name]):\n                            raise RuntimeError(\'Original weights were not restored\')\n            result = {\'case_id\': case_id, \'requested_rewrite\': record[\'requested_rewrite\'],\n                      \'run_key\': run_key, \'pre\': pre, \'post\': post, \'time\': edit_seconds,\n                      \'elapsed_seconds\': time.perf_counter()-started, \'weights_restored\': True}\n            validate_result(result, record, run_key)\n            atomic_json(output / f\'case_{case_id}.json\', result)\n            print(f\'Case {case_id} saved; original weights verified; {result["elapsed_seconds"]:.1f}s\', flush=True)\n    results = [validate_result(json.loads((output/f"case_{r[\'case_id\']}.json").read_text()), r, run_key)\n               for r in records[:args.limit]]\n    summary = aggregate(results)\n    summary[\'complete_200\'] = args.limit == 200 and len(results) == 200\n    summary[\'comparison_note\'] = \'Paper Table 4 uses 7,500 GPT-2 XL cases. This is a 200-case subset replication, not an exact full reproduction.\'\n    atomic_json(output / f\'summary_{args.limit}.json\', summary)\n    import csv\n    with (output / f\'comparison_{args.limit}.csv\').open(\'w\', newline=\'\') as f:\n        writer = csv.writer(f)\n        writer.writerow([\'metric\',\'pre_percent\',\'post_percent\',\'paper_percent\',\'difference_percentage_points\',\'n\'])\n        for metric in summary[\'post\']:\n            writer.writerow([metric, summary[\'pre\'][metric], summary[\'post\'][metric],\n                             summary[\'paper_gpt2_xl_7500_cases\'][metric], summary[\'difference_percentage_points\'][metric], len(results)])\n    print(json.dumps(summary, indent=2))\n\nif __name__ == \'__main__\':\n    main()\n', 'test_baseline_support.py': "import copy\nimport json\nimport tempfile\nimport unittest\nfrom pathlib import Path\nfrom baseline_support import aggregate, atomic_json, scores, validate_result\n\ndef fixture(case_id=1):\n    phase = {'rewrite_prompts_probs':[{'target_new':1.,'target_true':2.}],\n             'paraphrase_prompts_probs':[{'target_new':1.,'target_true':2.},{'target_new':2.,'target_true':1.}],\n             'neighborhood_prompts_probs':[{'target_new':2.,'target_true':1.}]}\n    return {'case_id':case_id,'requested_rewrite':{'subject':'X'}, 'run_key':'abc',\n            'weights_restored':True,'pre':copy.deepcopy(phase),'post':copy.deepcopy(phase)}\n\nclass ResultTests(unittest.TestCase):\n    def test_directions_and_ties(self):\n        result = fixture()\n        self.assertEqual(scores(result, 'post'), {'efficacy':100.,'paraphrase':50.,'neighborhood':100.})\n        result['post']['rewrite_prompts_probs'][0]['target_true'] = 1.\n        self.assertEqual(scores(result,'post')['efficacy'],0.)\n    def test_average_per_fact_not_pooled_prompts(self):\n        a,b = fixture(1),fixture(2)\n        b['post']['paraphrase_prompts_probs'] = [{'target_new':2.,'target_true':1.}]*8\n        self.assertEqual(aggregate([a,b])['post']['paraphrase'],25.)\n    def test_bad_resume_rejected(self):\n        r = fixture()\n        for change in [{'run_key':'other'}, {'weights_restored':False}, {'case_id':3}]:\n            broken = dict(r,**change)\n            with self.assertRaises(ValueError): validate_result(broken,r,'abc')\n    def test_invalid_metrics_rejected(self):\n        r = fixture()\n        r['post']['rewrite_prompts_probs'][0]['target_new'] = float('nan')\n        with self.assertRaises(ValueError): scores(r,'post')\n    def test_empty_and_duplicate_results(self):\n        with self.assertRaises(ValueError): aggregate([])\n        with self.assertRaises(ValueError): aggregate([fixture(),fixture()])\n    def test_atomic_save(self):\n        with tempfile.TemporaryDirectory() as d:\n            p = Path(d)/'case.json'\n            atomic_json(p, fixture())\n            self.assertEqual(json.loads(p.read_text()),fixture())\n            self.assertFalse(p.with_suffix('.json.tmp').exists())\n\nif __name__ == '__main__': unittest.main()\n"}
for name, source in EMBEDDED_FILES.items():
    (REPO / name).write_text(source)
subprocess.run([PYTHON, '-m', 'unittest', 'test_baseline_support', '-v'], cwd=REPO, check=True)

def run_stage(limit):
    # Stream progress into this cell. Any failure stops here; rerun to resume.
    subprocess.run([PYTHON, '-u', 'run_baseline.py', '--limit', str(limit), '--output', str(OUT)], cwd=REPO, check=True)
    summary = json.loads((OUT / f'summary_{limit}.json').read_text())
    assert summary['num_cases'] == limit
    print(f"Completed {limit} independent edits; weights restored after every case.")
    return summary


## Stage 1 — one fact
The first run downloads model weights, CounterFact and official covariance statistics. It can take much longer than subsequent runs. Inspect the output before continuing. Scores need not be perfect for the run to be valid.


In [11]:
smoke = run_stage(1)
print(json.dumps(smoke, indent=2))


Completed 1 independent edits; weights restored after every case.
{
  "num_cases": 1,
  "case_ids": [
    0
  ],
  "pre": {
    "efficacy": 0.0,
    "paraphrase": 100.0,
    "neighborhood": 100.0
  },
  "post": {
    "efficacy": 100.0,
    "paraphrase": 100.0,
    "neighborhood": 100.0
  },
  "paper_gpt2_xl_7500_cases": {
    "efficacy": 100.0,
    "paraphrase": 96.4,
    "neighborhood": 75.4
  },
  "difference_percentage_points": {
    "efficacy": 0.0,
    "paraphrase": 3.5999999999999943,
    "neighborhood": 24.599999999999994
  },
  "complete_200": false,
  "comparison_note": "Paper Table 4 uses 7,500 GPT-2 XL cases. This is a 200-case subset replication, not an exact full reproduction."
}


## Stage 2 — five facts
Previously saved cases are validated and reused. Check execution time and memory before the full run. A new process reloads the clean model; edits never accumulate.


In [12]:
pilot = run_stage(5)
case_times = [json.loads((OUT / f'case_{i}.json').read_text())['elapsed_seconds'] for i in pilot['case_ids']]
print('Median case seconds (including first-case initialization effects):', sorted(case_times)[len(case_times)//2])
print(json.dumps(pilot, indent=2))


Completed 5 independent edits; weights restored after every case.
Median case seconds (including first-case initialization effects): 18.994388974000003
{
  "num_cases": 5,
  "case_ids": [
    0,
    1,
    2,
    3,
    4
  ],
  "pre": {
    "efficacy": 20.0,
    "paraphrase": 30.0,
    "neighborhood": 66.0
  },
  "post": {
    "efficacy": 100.0,
    "paraphrase": 80.0,
    "neighborhood": 66.0
  },
  "paper_gpt2_xl_7500_cases": {
    "efficacy": 100.0,
    "paraphrase": 96.4,
    "neighborhood": 75.4
  },
  "difference_percentage_points": {
    "efficacy": 0.0,
    "paraphrase": -16.400000000000006,
    "neighborhood": -9.400000000000006
  },
  "complete_200": false,
  "comparison_note": "Paper Table 4 uses 7,500 GPT-2 XL cases. This is a 200-case subset replication, not an exact full reproduction."
}


## Stage 3 — 200 facts
This cell resumes from the existing five cases. If disconnected, reconnect with the same environment, run setup cells and rerun this stage. Configuration or GPU changes intentionally stop resumption; use a new run name to prevent mixed results.


In [13]:
assert (OUT / 'summary_5.json').exists(), 'Complete the five-fact pilot first.'
final_summary = run_stage(200)
assert final_summary['complete_200'] is True
assert len(list(OUT.glob('case_*.json'))) == 200
print('200-CASE BASELINE COMPLETE')
print(json.dumps(final_summary, indent=2))


Completed 200 independent edits; weights restored after every case.
200-CASE BASELINE COMPLETE
{
  "num_cases": 200,
  "case_ids": [
    0,
    1,
    2,
    3,
    4,
    5,
    6,
    7,
    8,
    9,
    10,
    11,
    12,
    13,
    14,
    15,
    16,
    17,
    18,
    19,
    20,
    21,
    22,
    23,
    24,
    25,
    26,
    27,
    28,
    29,
    30,
    31,
    32,
    33,
    34,
    35,
    36,
    37,
    38,
    39,
    40,
    41,
    42,
    43,
    44,
    45,
    46,
    47,
    48,
    49,
    50,
    51,
    52,
    53,
    54,
    55,
    56,
    57,
    58,
    59,
    60,
    61,
    62,
    63,
    64,
    65,
    66,
    67,
    68,
    69,
    70,
    71,
    72,
    73,
    74,
    75,
    76,
    77,
    78,
    79,
    80,
    81,
    82,
    83,
    84,
    85,
    86,
    87,
    88,
    89,
    90,
    91,
    92,
    93,
    94,
    95,
    96,
    97,
    98,
    99,
    100,
    101,
    102,
    103,
    104,
    105,
    106,
    107,
    1

## Export your handoff
The ZIP contains raw pre/post metrics, logs, configuration, package versions and comparison CSV. `complete_200` must be true before presenting the baseline as completed. The CSV is ready for Yuga's results table; the notebook setup can be reused by Soham and Nico.

Metrics are probability comparisons, averaged within each fact and then across facts. Fluency/generation consistency are explicitly omitted from this three-score assignment. See the paper for definitions.


In [14]:
assert json.loads((OUT / 'summary_200.json').read_text())['complete_200'] is True
archive = shutil.make_archive('/content/Group32_Rohith_ROME_Baseline_Results', 'zip', root_dir=OUT)
from google.colab import files
files.download(archive)
print('Saved:', archive)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Saved: /content/Group32_Rohith_ROME_Baseline_Results.zip
